FAST — NUCES, Karachi Campus

<div style="border-bottom:3px solid #1F3864;padding-bottom:10px">
<h1 style="color:#1F3864;margin-bottom:2px">Lab 05 — Object-Oriented Programming</h1>
<b>Programming for Artificial Intelligence</b><br>

Department of Artificial Intelligence &amp; Data Science<br>

<i>Prepared by Khadeejah Ashraf, Lab Instructor</i>
</div>

Read the explanation, run the code cell below it with **Shift + Enter**, and compare what you
get with what you expected. Cells marked **Your turn** are for you to fill in.

**Contents**

1. From functions to objects  
2. Classes and objects  
3. The constructor and instance attributes  
4. Methods and `self`  
5. Printing an object — `__str__` and friends  
6. Encapsulation  
7. Inheritance  
8. Method overriding and `super()`  
9. Polymorphism  
10. Abstract base classes  
11. Class Tasks (C1 – C5)  
12. Home Tasks


---
## 1. From Functions to Objects

In Lab 04 the data lived in a dictionary and the functions lived elsewhere. Nothing tied them
together, so a typo in a key went unnoticed.

In [ ]:
# procedural: the data and the code that understands it are separate
student = {'name': 'Ali', 'marks': [78, 45, 90]}

def average(record):
    return round(sum(record['marks']) / len(record['marks']), 2)

print(average(student))
student['mrks'] = []        # a typo nothing will catch until much later
print(student)

71.0
{'name': 'Ali', 'marks': [78, 45, 90], 'mrks': []}


In [ ]:
# object-oriented: one unit, and every object is built the same way
class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks

    def average(self):
        return round(sum(self.marks) / len(self.marks), 2)

ali = Student('Ali', [78, 45, 90])
print(ali.name, ali.average())      # the object already knows its own marks

Ali 71.0


---
## 2. Classes and Objects

A **class** is the blueprint; an **object** (instance) is one thing built from it. Calling the
class creates an object.

In [ ]:
class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks

ali  = Student('Ali', [78, 45, 90])
sara = Student('Sara', [88, 92, 95])

print(ali.name, sara.name)
print(type(ali))
print(isinstance(ali, Student))
print(ali is sara)          # two separate objects

Ali Sara
<class '__main__.Student'>
True
False


### 2.1 Class attributes vs instance attributes

An attribute in the class body is shared by every object. An attribute set through `self`
belongs to one object only.

In [ ]:
class Student:
    school = 'FAST NUCES'          # class attribute - shared
    count = 0                      # shared counter

    def __init__(self, name):
        self.name = name           # instance attribute - one per object
        Student.count += 1

ali  = Student('Ali')
sara = Student('Sara')

print(ali.school, sara.school)
print(Student.count)
print(ali.name, sara.name)

FAST NUCES FAST NUCES
2
Ali Sara


In [ ]:
# a trap: assigning through the OBJECT hides the class attribute, it does not change it
ali.school = 'Somewhere else'
print(ali.school, sara.school, Student.school)

Student.school = 'FAST NUCES Karachi'    # this is how you change the shared value
print(sara.school, Student.school)

Somewhere else FAST NUCES FAST NUCES
FAST NUCES Karachi FAST NUCES Karachi


**Your turn.** Write a class `Book` with `title`, `author` and `pages` set in the constructor,
create two books, and print each one's title and page count.

In [ ]:
# Your code here

---
## 3. The Constructor and Instance Attributes

`__init__` runs automatically on every new object. You never call it yourself, and it returns
nothing — its job is to set the object's starting attributes.

In [ ]:
class BankAccount:
    def __init__(self, holder, balance=0):
        self.holder = holder
        self.balance = balance
        self.history = []          # built inside __init__, so one list per object

a = BankAccount('Ali', 5000)
b = BankAccount('Sara')            # balance takes its default

print(a.holder, a.balance)
print(b.holder, b.balance)
print(a.history is b.history)      # separate lists, not one shared list

Ali 5000
Sara 0
False


In [ ]:
# the constructor is the right place to reject impossible data
class BankAccount:
    def __init__(self, holder, balance=0):
        if balance < 0:
            raise ValueError("balance cannot be negative")
        self.holder = holder
        self.balance = balance

try:
    BankAccount('Ali', -100)
except ValueError as err:
    print("rejected:", err)

rejected: balance cannot be negative


---
## 4. Methods and `self`

A method's first parameter is always `self` — the object it was called on. You write it in the
definition but never pass it at the call: `ali.average()` runs `Student.average(ali)`.

In [ ]:
class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks

    def average(self):                 # reads this object's data
        return round(sum(self.marks) / len(self.marks), 2)

    def add_mark(self, mark):          # changes this object's data
        self.marks.append(mark)

    def grade(self):                   # a method may call another method
        avg = self.average()
        if avg >= 80:
            return 'A'
        elif avg >= 70:
            return 'B'
        elif avg >= 60:
            return 'C'
        return 'F'

ali = Student('Ali', [78, 45, 90])
print(ali.average(), ali.grade())

ali.add_mark(100)
print(ali.marks, ali.average(), ali.grade())

print(Student.average(ali))            # the same call written the long way

71.0 B
[78, 45, 90, 100] 78.25 B
78.25


In [ ]:
# forgetting self - run it to see the message you will meet often
class Broken:
    def hello():
        print("hi")

try:
    Broken().hello()
except TypeError as err:
    print("TypeError:", err)

TypeError: Broken.hello() takes 0 positional arguments but 1 was given


### 4.1 Class methods and static methods

In [ ]:
class Student:
    count = 0

    def __init__(self, name):
        self.name = name
        Student.count += 1

    @classmethod
    def how_many(cls):                 # works on the class, not one object
        return cls.count

    @staticmethod
    def is_passing(mark):              # a helper that needs no object at all
        return mark >= 50

Student('Ali'); Student('Sara')
print(Student.how_many())
print(Student.is_passing(45), Student.is_passing(78))

2
False True


**Your turn.** Add to your `Book` class a method `is_long()` returning `True` when the book has
more than 300 pages, and a class attribute `library` shared by every book. Test both.

In [ ]:
# Your code here

---
## 5. Printing an Object — `__str__` and Friends

Methods whose names begin and end with two underscores hook your class into the language's own
syntax.

In [ ]:
class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks

plain = Student('Ali', [78, 45, 90])
print(plain)                 # unhelpful without __str__

In [ ]:
class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks

    def __str__(self):
        return "Student(" + self.name + ", " + str(len(self.marks)) + " marks)"

    def __len__(self):
        return len(self.marks)

    def __eq__(self, other):
        return self.name == other.name

ali   = Student('Ali', [78, 45, 90])
clone = Student('Ali', [10, 20])

print(ali)             # __str__
print(len(ali))        # __len__
print(ali == clone)    # __eq__

Student(Ali, 3 marks)
3
True


---
## 6. Encapsulation

| Written as | Meaning |
|---|---|
| `self.balance` | Public — anyone may read and change it |
| `self._bank` | Protected by convention — internal, but subclasses may use it |
| `self.__balance` | Private — the name is mangled to `_ClassName__balance` |

In [ ]:
class BankAccount:
    def __init__(self, holder, balance=0):
        self.holder = holder        # public
        self._bank = 'HBL'          # protected by convention
        self.__balance = balance    # private

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.__balance += amount

    def get_balance(self):
        return self.__balance

a = BankAccount('Ali', 5000)
a.deposit(1500)
print(a.holder, a.get_balance())
print(a._bank)                      # possible, but you are trespassing

try:
    print(a.__balance)
except AttributeError as err:
    print("AttributeError:", err)

Ali 6500
HBL
AttributeError: 'BankAccount' object has no attribute '__balance'


In [ ]:
# name mangling is protection against accidents, not security
print(a._BankAccount__balance)
print([n for n in vars(a)])

6500
['holder', '_bank', '_BankAccount__balance']


### 6.1 `@property` — attribute syntax with validation behind it

In [ ]:
class Student:
    def __init__(self, name, marks=0):
        self.name = name
        self.__marks = marks

    @property
    def marks(self):                 # reading obj.marks runs this
        return self.__marks

    @marks.setter
    def marks(self, value):          # assigning obj.marks = x runs this
        if not 0 <= value <= 100:
            raise ValueError("marks must be between 0 and 100")
        self.__marks = value

s = Student('Ali')
s.marks = 78
print(s.marks)

try:
    s.marks = 150
except ValueError as err:
    print("rejected:", err)

78
rejected: marks must be between 0 and 100


**Your turn.** Give your `Book` class a private `__price` attribute, a `price` property that
reads it, and a setter that refuses a negative price.

In [ ]:
# Your code here

---
## 7. Inheritance

A child class automatically gets every attribute and method of its parent and changes only what
differs. Look for the "is a" relationship: a `Bus` **is a** `Vehicle`.

In [ ]:
class Vehicle:
    def __init__(self, make, seats):
        self.make = make
        self.seats = seats

    def fare(self):
        return self.seats * 100

    def describe(self):
        return self.make + " with " + str(self.seats) + " seats"


class Bus(Vehicle):        # inherits everything
    pass                   # ...and adds nothing yet


b = Bus('Hino', 50)
print(b.describe())
print(b.fare())
print(isinstance(b, Bus), isinstance(b, Vehicle), issubclass(Bus, Vehicle))

Hino with 50 seats
5000
True True True


### 7.1 Extending the parent with `super()`

The child takes the extra values, hands the shared ones up to the parent, then sets what is new.

In [ ]:
class Bus(Vehicle):
    def __init__(self, make, seats, route):
        super().__init__(make, seats)     # let Vehicle set make and seats
        self.route = route                # then add what is new

    def describe(self):
        return super().describe() + " on route " + self.route


b = Bus('Hino', 50, 'Karachi-Hyderabad')
print(b.make, b.seats, b.route)
print(b.describe())

Hino 50 Karachi-Hyderabad
Hino with 50 seats on route Karachi-Hyderabad


### 7.2 Multilevel inheritance and the method resolution order

In [ ]:
class Machine:
    def __init__(self, make):
        self.make = make

class Car(Machine):
    def __init__(self, make, doors):
        super().__init__(make)
        self.doors = doors

class SportsCar(Car):
    def __init__(self, make, doors, top_speed):
        super().__init__(make, doors)
        self.top_speed = top_speed

s = SportsCar('Toyota', 2, 250)
print(s.make, s.doors, s.top_speed)
print([c.__name__ for c in SportsCar.__mro__])

Toyota 2 250
['SportsCar', 'Car', 'Machine', 'object']


In [ ]:
# multiple inheritance: the parents are searched LEFT TO RIGHT
class Engine:
    def start(self):
        return "engine started"

class Radio:
    def start(self):
        return "radio started"

class Hybrid(Engine, Radio):
    pass

class Hybrid2(Radio, Engine):     # the same parents, the other way round
    pass

print(Hybrid().start())
print(Hybrid2().start())
print([c.__name__ for c in Hybrid.__mro__])

engine started
radio started
['Hybrid', 'Engine', 'Radio', 'object']


---
## 8. Method Overriding and `super()`

The search for a method starts at the child, so a child's version wins over the parent's.

In [ ]:
class Vehicle:
    def __init__(self, make, seats):
        self.make, self.seats = make, seats

    def fare(self):
        return self.seats * 100


class Bus(Vehicle):
    def fare(self):                       # overrides Vehicle.fare completely
        return self.seats * 100 * 1.10


class Rickshaw(Vehicle):
    def fare(self):
        return self.seats * 60


for v in [Vehicle('Generic', 4), Bus('Hino', 50), Rickshaw('Sazgar', 3)]:
    print(type(v).__name__, v.fare())

Vehicle 400
Bus 5500.0
Rickshaw 180


In [ ]:
# an override can EXTEND rather than replace, by calling super()
class Bus(Vehicle):
    def fare(self):
        base = super().fare()             # reuse Vehicle's calculation
        return base + base * 0.10         # then add the bus-specific charge

print(Bus('Hino', 50).fare())

5500.0


**Your turn.** Derive `SchoolBus` from `Bus` with a fixed fare of `2500` regardless of the
number of seats, create one, print its fare, and print `SchoolBus.__mro__`.

In [ ]:
# Your code here

---
## 9. Polymorphism

The same call means different things depending on the object it lands on. You have relied on
this since Lab 01.

In [ ]:
for value in ["abc", [1, 2, 3, 4], {'a': 1}]:
    print(type(value).__name__, len(value))

print(3 + 4)
print("Prog" + "ramming")

str 3
list 4
dict 1
7
Programming


In [ ]:
# duck typing: these three classes share no parent at all
class Car:
    def describe(self):
        return "A car carries a small family."

class Truck:
    def describe(self):
        return "A truck carries cargo."

class Rickshaw3:
    def describe(self):
        return "A rickshaw carries three passengers."

for vehicle in [Car(), Truck(), Rickshaw3()]:
    print(vehicle.describe())      # nothing here asks what type it is

A car carries a small family.
A truck carries cargo.
A rickshaw carries three passengers.


In [ ]:
# one function, any class that has make and fare()
def print_details(vehicle):
    """Work with any object that has a make and a fare()."""
    print(vehicle.make, "->", vehicle.fare())

print_details(Bus('Hino', 50))
print_details(Rickshaw('Sazgar', 3))

Hino -> 5500.0
Sazgar -> 180


### 9.1 Operator overloading

In [ ]:
class Money:
    def __init__(self, rupees):
        self.rupees = rupees

    def __add__(self, other):
        return Money(self.rupees + other.rupees)

    def __str__(self):
        return "Rs " + str(self.rupees)

print(Money(1500) + Money(2500))

Rs 4000


In [ ]:
# Python has no compile-time overloading: the second definition replaces the first
def product(a, b):
    return a * b

def product(a, b, c):
    return a * b * c

try:
    print(product(4, 5))
except TypeError as err:
    print("TypeError:", err)

print(product(4, 5, 5))     # only the last definition survives

TypeError: product() missing 1 required positional argument: 'c'
100


---
## 10. Abstract Base Classes

A parent that exists only to state what its children must provide. A class deriving from `ABC`
with an `@abstractmethod` cannot be instantiated, and neither can a subclass that fails to
implement it.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def area(self):
        """Every concrete shape must supply this."""

    def describe(self):                 # a concrete method, inherited as usual
        return self.name + " has area " + str(round(self.area(), 2))


class Rectangle(Shape):
    def __init__(self, width, height):
        super().__init__('Rectangle')
        self.width, self.height = width, height

    def area(self):
        return self.width * self.height


class Circle(Shape):
    def __init__(self, radius):
        super().__init__('Circle')
        self.radius = radius

    def area(self):
        return 3.14159 * self.radius ** 2


for shape in [Rectangle(4, 5), Circle(3)]:
    print(shape.describe())

Rectangle has area 20
Circle has area 28.27


In [ ]:
# the abstract class itself cannot be instantiated
try:
    Shape('generic')
except TypeError as err:
    print("TypeError:", err)

# nor can a subclass that forgets to implement area()
class Hexagon(Shape):
    def __init__(self, side):
        super().__init__('Hexagon')
        self.side = side

try:
    Hexagon(4)
except TypeError as err:
    print("TypeError:", err)

TypeError: Can't instantiate abstract class Shape without an implementation for abstract method 'area'
TypeError: Can't instantiate abstract class Hexagon without an implementation for abstract method 'area'


---
# 11. Class Tasks

Complete all five during the lab session and demonstrate them to the instructor before you
leave. Every class needs a docstring, every attribute must be set in the constructor, and only
a display method may call `print()`.

### C1 — A Book Class
*classes · constructors · attributes · methods*

Write a class `Book` with the class attribute `library = 'FAST NUCES Library'` and the instance
attributes `title`, `author`, `pages` and `issued` (starting as `False`). Give it:

* `issue()` — marks the book as issued and returns a message, or refuses if it is already out
* `return_book()` — reverses that
* `is_long()` — `True` when the book has more than 300 pages
* `__str__` — printing a book shows its title, author and current status

Create three books, issue two of them, try to issue one of those a second time, and print all
three. Then add a class attribute `count` recording how many books have been created, and print
it through a `@classmethod` called `how_many()`.

Finish with one line showing that changing `Book.library` changes it for every existing object,
and a comment explaining why.

In [ ]:
# C1 - A Book Class
# Name:
# Roll number:

class Book:
    """Represents a book in the library system."""
    library = 'FAST NUCES Library'  # Class attribute
    count = 0  # Class attribute to count books

    def __init__(self, title, author, pages):
        self.title = title
        self.author = author
        self.pages = pages
        self.issued = False  # Books start as not issued
        Book.count += 1

    def issue(self):
        """Marks the book as issued if not already."""
        if self.issued:
            return f"'{self.title}' is already issued."
        else:
            self.issued = True
            return f"'{self.title}' has been issued."

    def return_book(self):
        """Marks the book as returned if issued."""
        if not self.issued:
            return f"'{self.title}' is not currently issued."
        else:
            self.issued = False
            return f"'{self.title}' has been returned."

    def is_long(self):
        """Returns True if the book has more than 300 pages."""
        return self.pages > 300

    def __str__(self):
        """Returns a string representation of the book's status."""
        status = "Issued" if self.issued else "Available"
        return f"Book('{self.title}' by {self.author}, {self.pages} pages, Status: {status})"

    @classmethod
    def how_many(cls):
        """Returns the total number of books created."""
        return cls.count

# Driver code for C1

# Create three books
book1 = Book("The Lord of the Rings", "J.R.R. Tolkien", 1178)
book2 = Book("Python Crash Course", "Eric Matthes", 544)
book3 = Book("The Hitchhiker's Guide to the Galaxy", "Douglas Adams", 193)

# Issue two of them
print(book1.issue())
print(book2.issue())

# Try to issue one of those a second time
print(book1.issue())

# Print all three books
print("\n--- Current Book Status ---")
print(book1)
print(book2)
print(book3)

# Demonstrate is_long()
print(f"\nIs '{book1.title}' a long book? {book1.is_long()}")
print(f"Is '{book3.title}' a long book? {book3.is_long()}")

# Print the count of books through a @classmethod
print(f"\nTotal number of books created: {Book.how_many()}")

# Demonstrate changing Book.library and explaining why
print(f"\nOriginal library for book1: {book1.library}")
Book.library = 'New Central Library' # Changing the class attribute
print(f"New library for book1: {book1.library}")
print(f"New library for book2: {book2.library}")
# Explanation: Changing Book.library changes it for every existing object because 'library' is a class attribute.
# All instances of the class share the same class attribute, so modifying it through the class name
# affects all objects that access this attribute without having their own instance-specific version of it.

### C2 — A Bank Account
*encapsulation*

`__init__` takes the holder's name, a starting balance (default `0`) and a four-digit security
code. The balance and the security code must be **private**; the account number **protected**.
Provide:

* `deposit(amount)` and `withdraw(amount)`, raising `ValueError` on a non-positive amount or an
  overdraw
* a `balance` `@property` that allows reading but not direct assignment
* `statement()` — prints the holder, the account number and the last five transactions from a
  private history list
* `verify(code)` — returns `True` or `False` without ever revealing the stored code

Demonstrate that `account.__balance` raises `AttributeError` from outside the class, then that
`account._BankAccount__balance` reaches it anyway, and explain in a comment what that tells you
about privacy in Python.

In [ ]:
# C2 - A Bank Account

import random

class BankAccount:
    """Represents a bank account with encapsulation for balance and security code."""

    def __init__(self, holder, balance=0, security_code='0000'):
        if not isinstance(holder, str) or not holder.strip():
            raise ValueError("Account holder name cannot be empty.")
        if not isinstance(balance, (int, float)) or balance < 0:
            raise ValueError("Initial balance cannot be negative.")
        if not isinstance(security_code, str) or not (len(security_code) == 4 and security_code.isdigit()):
            raise ValueError("Security code must be a four-digit string.")

        self.holder = holder  # Public attribute
        self._account_number = ''.join(random.choices('0123456789', k=10)) # Protected attribute
        self.__balance = balance  # Private attribute
        self.__security_code = security_code  # Private attribute
        self._history = []  # Private-like protected attribute for transaction history

        if balance > 0:
            self._history.append(f"Initial deposit: {balance:.2f}")

    def deposit(self, amount):
        """Deposits money into the account."""
        if not isinstance(amount, (int, float)) or amount <= 0:
            raise ValueError("Deposit amount must be positive.")
        self.__balance += amount
        self._history.append(f"Deposit: {amount:.2f}")
        return f"Deposited {amount:.2f}. New balance: {self.__balance:.2f}"

    def withdraw(self, amount):
        """Withdraws money from the account."""
        if not isinstance(amount, (int, float)) or amount <= 0:
            raise ValueError("Withdrawal amount must be positive.")
        if amount > self.__balance:
            raise ValueError("Insufficient funds.")
        self.__balance -= amount
        self._history.append(f"Withdrawal: {amount:.2f}")
        return f"Withdrew {amount:.2f}. New balance: {self.__balance:.2f}"

    @property
    def balance(self):
        """Allows reading the account balance (read-only property)."""
        return self.__balance

    def statement(self):
        """Prints the account holder, account number, and last five transactions."""
        print(f"\n--- Account Statement for {self.holder} ---")
        print(f"Account Number: {self._account_number}")
        print(f"Current Balance: {self.__balance:.2f}")
        print("--- Recent Transactions ---")
        if not self._history:
            print("No transactions yet.")
        else:
            for entry in self._history[-5:]:
                print(entry)
        print("------------------------------------")

    def verify(self, code):
        """Verifies the provided security code."""
        return code == self.__security_code

# Driver code for C2

# Create a bank account
account1 = BankAccount('Ali Raza', 1000, '1234')

print(f"Account holder: {account1.holder}")
print(f"Account number: {account1._account_number}") # Accessing protected attribute
print(f"Initial balance: {account1.balance:.2f}") # Accessing balance via property

# Demonstrate deposit and withdraw
print(account1.deposit(500))
print(account1.withdraw(200))

# Try to withdraw more than balance
try:
    print(account1.withdraw(2000))
except ValueError as e:
    print(f"Error during withdrawal: {e}")

# Try to deposit a negative amount
try:
    print(account1.deposit(-100))
except ValueError as e:
    print(f"Error during deposit: {e}")

# Print statement
account1.statement()

# Demonstrate security code verification
print(f"Is '1234' the correct security code? {account1.verify('1234')}")
print(f"Is '0000' the correct security code? {account1.verify('0000')}")

# Demonstrate that account.__balance raises AttributeError from outside the class
try:
    print(account1.__balance)
except AttributeError as err:
    print(f"\nAttributeError: {err}")

# Demonstrate that account._BankAccount__balance reaches it anyway
print(f"Accessing __balance directly via name mangling: {account1._BankAccount__balance:.2f}")

# Explanation of Python's 'privacy'
# Explanation: In Python, attributes prefixed with double underscores (e.g., `__balance`) are name-mangled.
# This means the interpreter renames them to `_ClassName__attribute` (e.g., `_BankAccount__balance`).
# This mechanism is primarily for preventing accidental name clashes in subclasses, not for strict security.
# It provides a weak form of encapsulation, as the 'private' attribute can still be accessed directly
# if one knows the mangled name, demonstrating that Python relies on convention rather than strict enforcement
# for data hiding. Attributes prefixed with a single underscore (e.g., `_account_number`) are conventions
# indicating they should be treated as protected and not directly accessed from outside the class or its subclasses.

### C3 — A Vehicle Hierarchy
*inheritance · overriding · super()*

Base class `Vehicle` with `make`, `model`, `seats`, `fare()` returning `seats * 100`, and
`describe()`. Derive:

* `Bus` — overrides `fare()` to add a 10% maintenance charge
* `Car` — adds an `ac` attribute and adds 25% when the air conditioning is on
* `Truck` — replaces `seats` with `load_tonnes` and charges `load_tonnes * 500`

Every override of `describe()` must call `super().describe()` instead of repeating it. Put four
vehicles in a list, loop over it once, and print each description and fare. Then derive
`SchoolBus` from `Bus` with a fixed monthly fare, print `SchoolBus.__mro__`, and state in a
comment which class supplies `describe()` for a `SchoolBus` object and why.

In [ ]:
# C3 - A Vehicle Hierarchy

class Vehicle:
    """Base class for all vehicles."""
    def __init__(self, make, model, seats):
        self.make = make
        self.model = model
        self.seats = seats

    def fare(self):
        """Calculates the base fare based on seats."""
        return self.seats * 100

    def describe(self):
        """Returns a string description of the vehicle."""
        return f"{self.make} {self.model} with {self.seats} seats"


class Bus(Vehicle):
    """Represents a bus, inheriting from Vehicle, with a maintenance charge."""
    def __init__(self, make, model, seats):
        super().__init__(make, model, seats)

    def fare(self):
        """Overrides fare to add a 10% maintenance charge."""
        return super().fare() * 1.10

    # describe() is inherited from Vehicle, as no bus-specific additions are mentioned.


class Car(Vehicle):
    """Represents a car, inheriting from Vehicle, with optional AC."""
    def __init__(self, make, model, seats, ac=False):
        super().__init__(make, model, seats)
        self.ac = ac

    def fare(self):
        """Overrides fare to add a 25% charge if AC is on."""
        base_fare = super().fare()
        if self.ac:
            return base_fare * 1.25
        return base_fare

    def describe(self):
        """Overrides describe to include AC status."""
        ac_status = "(AC: On)" if self.ac else "(AC: Off)"
        return f"{super().describe()} {ac_status}"


class Truck(Vehicle):
    """Represents a truck, replacing seats with load_tonnes."""
    def __init__(self, make, model, load_tonnes):
        # Trucks don't primarily use 'seats' for their fare calculation or description,
        # but the Vehicle constructor expects it. We pass 0 and override its usage.
        super().__init__(make, model, seats=0) # Pass 0 for seats as it's not relevant for truck fare
        self.load_tonnes = load_tonnes

    def fare(self):
        """Overrides fare to calculate based on load tonnes."""
        return self.load_tonnes * 500

    def describe(self):
        """Overrides describe to use load capacity instead of seats."""
        return f"{self.make} {self.model} with {self.load_tonnes} tonnes load capacity"


class SchoolBus(Bus):
    """Represents a school bus with a fixed monthly fare."""
    def __init__(self, make, model, seats):
        super().__init__(make, model, seats)

    def fare(self):
        """Overrides fare with a fixed monthly rate."""
        return 2500 # Fixed fare


# Driver code for C3

# Create four vehicles
vehicles = [
    Bus('Hino', 'Bluebird', 50),
    Car('Honda', 'Civic', 4, ac=True),
    Truck('Ford', 'F-150', 2.5),
    Car('Toyota', 'Corolla', 5, ac=False)
]

print("\n--- Vehicle Descriptions and Fares ---")
for vehicle in vehicles:
    print(f"Type: {type(vehicle).__name__}, Description: {vehicle.describe()}, Fare: {vehicle.fare():.2f}")

# Create a SchoolBus
school_bus = SchoolBus('School Transport', 'Sunshine', 30)
print(f"\nSchool Bus Description: {school_bus.describe()}, Fare: {school_bus.fare():.2f}")

# Print SchoolBus.__mro__
print("\nSchoolBus Method Resolution Order:", [cls.__name__ for cls in SchoolBus.__mro__])

# Explanation for SchoolBus describe()
# The describe() method for a SchoolBus object is supplied by the Vehicle class.
# This is because SchoolBus inherits from Bus, and Bus does not override the describe() method.
# Therefore, when describe() is called on a SchoolBus instance, Python's Method Resolution Order (MRO)
# looks for the method first in SchoolBus, then in Bus, and then in Vehicle. It finds and uses
# the describe() method from the Vehicle class, as it is the first concrete implementation in the MRO.

### C4 — Shapes
*abstraction · polymorphism*

Using `ABC` and `@abstractmethod`, write an abstract `Shape` with abstract `area()` and
`perimeter()` and a concrete `summary()` that uses both. Implement `Rectangle`, `Square`
(derived from `Rectangle`, **not** from `Shape`), `Triangle` (three sides, Heron's formula) and
`Circle`. Store one of each in a list, print every summary with a single loop, then print the
shapes sorted from the largest area to the smallest using `sorted()` with a `key`.

Show — inside `try` / `except` so the notebook still runs to the end — that `Shape()` raises a
`TypeError`, and that a subclass which forgets `perimeter()` cannot be instantiated either.

In [ ]:
# C4 - Shapes

from abc import ABC, abstractmethod
import math

class Shape(ABC):
    """Abstract base class for all shapes."""
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def area(self):
        """Abstract method to calculate the area of the shape."""
        pass

    @abstractmethod
    def perimeter(self):
        """Abstract method to calculate the perimeter of the shape."""
        pass

    def summary(self):
        """Returns a summary string including name, area, and perimeter."""
        return f"{self.name}: Area = {self.area():.2f}, Perimeter = {self.perimeter():.2f}"


class Rectangle(Shape):
    """Represents a rectangle, inheriting from Shape."""
    def __init__(self, width, height):
        super().__init__("Rectangle")
        if width <= 0 or height <= 0:
            raise ValueError("Width and height must be positive.")
        self.width = width
        self.height = height

    def area(self):
        """Calculates the area of the rectangle."""
        return self.width * self.height

    def perimeter(self):
        """Calculates the perimeter of the rectangle."""
        return 2 * (self.width + self.height)


class Square(Rectangle):
    """Represents a square, inheriting from Rectangle."""
    def __init__(self, side):
        super().__init__(side, side) # Initialize Rectangle with equal sides
        self.name = "Square" # Override name to be 'Square'

    # area() and perimeter() are inherited correctly from Rectangle


class Triangle(Shape):
    """Represents a triangle, using Heron's formula for area."""
    def __init__(self, s1, s2, s3):
        super().__init__("Triangle")
        if not (s1 + s2 > s3 and s1 + s3 > s2 and s2 + s3 > s1):
            raise ValueError("Invalid side lengths for a triangle.")
        self.s1 = s1
        self.s2 = s2
        self.s3 = s3

    def area(self):
        """Calculates the area of the triangle using Heron's formula."""
        s = self.perimeter() / 2  # Semi-perimeter
        return math.sqrt(s * (s - self.s1) * (s - self.s2) * (s - self.s3))

    def perimeter(self):
        """Calculates the perimeter of the triangle."""
        return self.s1 + self.s2 + self.s3


class Circle(Shape):
    """Represents a circle, inheriting from Shape."""
    def __init__(self, radius):
        super().__init__("Circle")
        if radius <= 0:
            raise ValueError("Radius must be positive.")
        self.radius = radius

    def area(self):
        """Calculates the area of the circle."""
        return math.pi * (self.radius ** 2)

    def perimeter(self):
        """Calculates the circumference (perimeter) of the circle."""
        return 2 * math.pi * self.radius


# Driver code for C4

# Create various shapes
shapes = [
    Rectangle(4, 5),
    Square(6),
    Triangle(3, 4, 5), # A right-angled triangle
    Circle(3)
]

# Print every summary with a single loop (polymorphism)
print("\n--- Shape Summaries ---")
for shape in shapes:
    print(shape.summary())

# Print the shapes sorted from the largest area to the smallest using sorted() with a key
print("\n--- Shapes Sorted by Area (Largest to Smallest) ---")
sorted_shapes = sorted(shapes, key=lambda s: s.area(), reverse=True)
for shape in sorted_shapes:
    print(shape.summary())

# Demonstrate that Shape() raises a TypeError
print("\n--- Abstract Class Instantiation Error ---")
try:
    Shape('generic')
except TypeError as err:
    print("TypeError:", err)

# Demonstrate that a subclass which forgets perimeter() cannot be instantiated either
class IncompleteShape(Shape):
    def __init__(self):
        super().__init__("Incomplete")

    def area(self):
        return 0 # Only implement area

try:
    IncompleteShape()
except TypeError as err:
    print("TypeError (IncompleteShape):", err)


### C5 — A Small Library System
*the whole lab in one program*

Three classes:

* `LibraryItem` — abstract base with `title`, a private `__item_id`, an `is_available` flag, an
  abstract `loan_period_days()` and a concrete `describe()`
* `PrintedBook` and `DVD` — derived, with different loan periods and their own extra attributes
* `Member` — a name, a private contact number and a list of borrowed items, with `borrow(item)`
  and `give_back(item)` that update both the member and the item and refuse to borrow something
  already out, plus a `history()` method

Finally write one function `show(entity)` that prints the details of **any** of these objects —
book, DVD or member — to demonstrate polymorphism. Your driver code must create at least two
members and four items, perform a borrow, a refused borrow and a return, and end by printing
every member's history.

In [ ]:
# C5 - A Small Library System

from abc import ABC, abstractmethod
import random

class LibraryItem(ABC):
    """Abstract base class for all library items."""
    def __init__(self, title):
        self.title = title
        self.__item_id = self._generate_item_id() # Private item ID
        self.is_available = True

    def _generate_item_id(self):
        """Generates a unique item ID."""
        return f"{self.title[:3].upper()}-{random.randint(1000, 9999)}"

    @abstractmethod
    def loan_period_days(self):
        """Abstract method to return the loan period in days."""
        pass

    def describe(self):
        """Returns a string description of the library item."""
        status = "Available" if self.is_available else "On Loan"
        return f"Title: '{self.title}' (ID: {self.__item_id}), Status: {status}, Loan Period: {self.loan_period_days()} days"


class PrintedBook(LibraryItem):
    """Represents a printed book, derived from LibraryItem."""
    def __init__(self, title, author, isbn):
        super().__init__(title)
        self.author = author
        self.isbn = isbn

    def loan_period_days(self):
        return 21  # 3 weeks

    def describe(self):
        base_desc = super().describe()
        return f"Book: '{self.title}' by {self.author} (ISBN: {self.isbn}), {base_desc[base_desc.find('(ID'):]}"


class DVD(LibraryItem):
    """Represents a DVD, derived from LibraryItem."""
    def __init__(self, title, director, duration_minutes):
        super().__init__(title)
        self.director = director
        self.duration_minutes = duration_minutes

    def loan_period_days(self):
        return 7  # 1 week

    def describe(self):
        base_desc = super().describe()
        return f"DVD: '{self.title}' directed by {self.director} ({self.duration_minutes} mins), {base_desc[base_desc.find('(ID'):]}"


class Member:
    """Represents a library member."""
    def __init__(self, name, contact_number):
        self.name = name
        self.__contact_number = contact_number # Private contact number
        self.borrowed_items = []
        self.history = [] # List of (item_title, action)

    def borrow(self, item):
        """Allows a member to borrow a library item."""
        if not isinstance(item, LibraryItem):
            print(f"Error: '{self.name}' cannot borrow an invalid item type.")
            return False

        if not item.is_available:
            print(f"'{self.name}' cannot borrow '{item.title}': Item is currently on loan.")
            self.history.append((item.title, "Failed Borrow: Item on loan"))
            return False

        for borrowed_item in self.borrowed_items:
            if borrowed_item.__item_id == item.__item_id: # Accessing private ID for unique check
                print(f"'{self.name}' already has '{item.title}' borrowed.")
                self.history.append((item.title, "Failed Borrow: Already borrowed"))
                return False

        item.is_available = False
        self.borrowed_items.append(item)
        self.history.append((item.title, "Borrowed"))
        print(f"'{self.name}' has successfully borrowed '{item.title}'.")
        return True

    def give_back(self, item):
        """Allows a member to return a borrowed item."""
        if item in self.borrowed_items:
            item.is_available = True
            self.borrowed_items.remove(item)
            self.history.append((item.title, "Returned"))
            print(f"'{self.name}' has successfully returned '{item.title}'.")
            return True
        else:
            print(f"'{self.name}' cannot return '{item.title}': Item not found in their borrowed list.")
            self.history.append((item.title, "Failed Return: Not borrowed by member"))
            return False

    def display_history(self):
        """Prints the member's borrowing and returning history."""
        print(f"\n--- {self.name}'s Transaction History ---")
        if not self.history:
            print("No transactions yet.")
        else:
            for title, action in self.history:
                print(f"  - '{title}': {action}")
        print("------------------------------------------")

    def describe(self):
        """Returns a string description of the member."""
        borrowed_titles = [item.title for item in self.borrowed_items]
        return f"Member: {self.name}, Contact: (Private), Borrowed: {', '.join(borrowed_titles) if borrowed_titles else 'None'}"


def show(entity):
    """Prints the details of any LibraryItem or Member object (polymorphic)."""
    if hasattr(entity, 'describe') and callable(getattr(entity, 'describe')):
        print(entity.describe())
    else:
        print("Cannot display details for this entity.")


# Driver code for C5

# Create Library Items
book1 = PrintedBook("The Great Gatsby", "F. Scott Fitzgerald", "978-0743273565")
book2 = PrintedBook("1984", "George Orwell", "978-0451524935")
dvd1 = DVD("Inception", "Christopher Nolan", 148)
dvd2 = DVD("The Matrix", "The Wachowskis", 136)

# Create Members
member1 = Member("Ali Khan", "03001234567")
member2 = Member("Sara Ahmed", "03339876543")

print("--- Initial Item Status ---")
show(book1)
show(dvd1)

# Demonstrate borrowing
print("\n--- Borrowing Actions ---")
member1.borrow(book1)
member2.borrow(dvd1)

# Attempt to borrow an item already out
member2.borrow(book1)

# Attempt to borrow an item already borrowed by self (should be handled by borrow logic)
member1.borrow(book1)

# Demonstrate returning
print("\n--- Returning Actions ---")
member1.give_back(book1)

# Attempt to return an item not borrowed by member
member2.give_back(book2)

# Verify item status after transactions
print("\n--- Item Status After Transactions ---")
show(book1)
show(dvd1)

# Display all members' history
member1.display_history()
member2.display_history()

# Demonstrate show() function for various entities
print("\n--- Polymorphic 'show' function demonstration ---")
show(book2)
show(dvd2)
show(member1)
show(member2)

# Demonstrate accessing private contact number (should fail or show mangled name)
print("\n--- Accessing Private Member Contact ---")
try:
    print(f"Member 1 contact: {member1.__contact_number}")
except AttributeError as err:
    print(f"AttributeError: {err}")

# Accessing via name mangling
print(f"Accessing contact via name mangling: {member1._Member__contact_number}")


## 12. Home Tasks

### Home Task 1 — Rectangle Class

In [ ]:
class Rectangle:
    """Represents a rectangle with length and width."""
    def __init__(self, length, width):
        if length <= 0 or width <= 0:
            raise ValueError("Length and width must be positive.")
        self.length = length
        self.width = width

    def area(self):
        """Calculates and returns the area of the rectangle."""
        return self.length * self.width

    def perimeter(self):
        """Calculates and returns the perimeter of the rectangle."""
        return 2 * (self.length + self.width)

    def __str__(self):
        """Returns a string representation of the rectangle."""
        return f"Rectangle(Length: {self.length}, Width: {self.width})"

# Driver code for Home Task 1

# Create three Rectangle objects
rect1 = Rectangle(10, 5)
rect2 = Rectangle(7, 7)
rect3 = Rectangle(12.5, 4)

# Print each with its area and perimeter
print("--- Rectangle Details ---")
for i, rect in enumerate([rect1, rect2, rect3], 1):
    print(f"Rectangle {i}: {rect}")
    print(f"  Area: {rect.area():.2f}")
    print(f"  Perimeter: {rect.perimeter():.2f}\n")

### Home Task 2 — Student Class

In [ ]:
class Student:
    """Represents a student with roll number, name, and three marks."""
    def __init__(self, roll, name, mark1, mark2, mark3):
        if not isinstance(roll, str) or not roll.strip():
            raise ValueError("Roll number cannot be empty.")
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Student name cannot be empty.")
        if not all(0 <= mark <= 100 for mark in [mark1, mark2, mark3]):
            raise ValueError("Marks must be between 0 and 100.")

        self.roll = roll
        self.name = name
        self.mark1 = mark1
        self.mark2 = mark2
        self.mark3 = mark3

    def total(self):
        """Calculates the sum of all marks."""
        return self.mark1 + self.mark2 + self.mark3

    def average(self):
        """Calculates the average of all marks."""
        return round(self.total() / 3, 2)

    def grade(self):
        """Determines the grade based on the average mark."""
        avg = self.average()
        if avg >= 90:
            return 'A'
        elif avg >= 80:
            return 'B'
        elif avg >= 70:
            return 'C'
        elif avg >= 60:
            return 'D'
        else:
            return 'F'

    def display(self):
        """Prints the student's details, total, average, and grade."""
        print(f"\n--- Student Details ---")
        print(f"Roll No: {self.roll}")
        print(f"Name: {self.name}")
        print(f"Marks: {self.mark1}, {self.mark2}, {self.mark3}")
        print(f"Total Marks: {self.total()}")
        print(f"Average Marks: {self.average()}")
        print(f"Grade: {self.grade()}")

    def __str__(self):
        """Returns a string representation of the student."""
        return f"Student(Roll: {self.roll}, Name: {self.name}, Avg: {self.average()})"

student1 = Student('FA19-BCS-001', 'Ali', 85, 90, 88)
student2 = Student('FA19-BCS-002', 'Sara', 70, 75, 68)
student3 = Student('FA19-BCS-003', 'Ahmed', 92, 95, 90)
student4 = Student('FA19-BCS-004', 'Fatima', 60, 65, 72)

students = [student1, student2, student3, student4]

print("--- All Students' Details ---")
for student in students:
    student.display()

# Find and print the topper's name
topper = None
max_average = -1

for student in students:
    if student.average() > max_average:
        max_average = student.average()
        topper = student

if topper:
    print(f"\n--- Topper Details ---")
    print(f"The topper is: {topper.name} with an average of {topper.average():.2f}")
else:
    print("No students to determine a topper.")

### Home Task 3 — Circle Class

In [ ]:
import math

class Circle:
    """Represents a circle with a private radius attribute."""
    def __init__(self, radius):
        self.__radius = 0 # Initialize to 0, then use the setter for validation
        self.radius = radius # Use the setter for initial validation

    @property
    def radius(self):
        """Getter for the private radius attribute."""
        return self.__radius

    @radius.setter
    def radius(self, value):
        """Setter for the radius, rejecting negative values."""
        if not isinstance(value, (int, float)) or value <= 0:
            raise ValueError("Radius must be a positive number.")
        self.__radius = value

    def area(self):
        """Calculates and returns the area of the circle."""
        return math.pi * (self.radius ** 2)

    def circumference(self):
        """Calculates and returns the circumference of the circle."""
        return 2 * math.pi * self.radius

    def __str__(self):
        """Returns a string representation of the circle."""
        return f"Circle(Radius: {self.radius:.2f})

circle1 = Circle(5)
print(f"Valid Circle: {circle1}")
print(f"  Area: {circle1.area():.2f}")
print(f"  Circumference: {circle1.circumference():.2f}\n")

# Demonstrate valid assignment via setter
circle1.radius = 7.5
print(f"Updated Circle: {circle1}")
print(f"  Area: {circle1.area():.2f}")
print(f"  Circumference: {circle1.circumference():.2f}\n")

print("Attempting to set invalid radius:")
try:
    circle1.radius = -2
except ValueError as e:
    print(f"  Error: {e}")

print("\nAttempting to create Circle with invalid initial radius:")
try:
    circle_invalid = Circle(0)
except ValueError as e:
    print(f"  Error: {e}")

In [ ]:
# Calculate area and circumference for a new Circle instance

new_circle = Circle(radius=10)
print(f"\nNew Circle: {new_circle}")
print(f"  Area: {new_circle.area():.2f}")
print(f"  Circumference: {new_circle.circumference():.2f}")

In [ ]:
# Create another Circle instance with a specific radius

circle_specific_radius = Circle(radius=8)
print(f"\nCircle with specific radius: {circle_specific_radius}")
print(f"  Area: {circle_specific_radius.area():.2f}")
print(f"  Circumference: {circle_specific_radius.circumference():.2f}")

In [ ]:
import unittest
import math

class TestCircle(unittest.TestCase):
    """Unit tests for the Circle class."""

    def test_init_valid_radius(self):
        """Test Circle initialization with a valid positive radius."""
        c = Circle(5)
        self.assertEqual(c.radius, 5)

    def test_init_invalid_radius_zero(self):
        """Test Circle initialization with zero radius should raise ValueError."""
        with self.assertRaises(ValueError):
            Circle(0)

    def test_init_invalid_radius_negative(self):
        """Test Circle initialization with a negative radius should raise ValueError."""
        with self.assertRaises(ValueError):
            Circle(-5)

    def test_radius_setter_valid(self):
        """Test setting radius to a valid positive value."""
        c = Circle(1)
        c.radius = 10
        self.assertEqual(c.radius, 10)

    def test_radius_setter_invalid_zero(self):
        """Test setting radius to zero should raise ValueError."""
        c = Circle(1)
        with self.assertRaises(ValueError):
            c.radius = 0

    def test_radius_setter_invalid_negative(self):
        """Test setting radius to a negative value should raise ValueError."""
        c = Circle(1)
        with self.assertRaises(ValueError):
            c.radius = -10

    def test_area(self):
        """Test the area calculation for a known radius."""
        c = Circle(5)
        expected_area = math.pi * (5 ** 2)
        self.assertAlmostEqual(c.area(), expected_area, places=7)

        c2 = Circle(1)
        expected_area2 = math.pi * (1 ** 2)
        self.assertAlmostEqual(c2.area(), expected_area2, places=7)

    def test_circumference(self):
        """Test the circumference calculation for a known radius."""
        c = Circle(5)
        expected_circumference = 2 * math.pi * 5
        self.assertAlmostEqual(c.circumference(), expected_circumference, places=7)

        c2 = Circle(1)
        expected_circumference2 = 2 * math.pi * 1
        self.assertAlmostEqual(c2.circumference(), expected_circumference2, places=7)

    def test_str(self):
        """Test the string representation of the Circle object."""
        c = Circle(3.14159)
        self.assertEqual(str(c), "Circle(Radius: 3.14)")



suite = unittest.TestSuite()
suite.addTest(unittest.makeSuite(TestCircle))
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)
print("------------------------------------------")

---
# 12. Home Tasks

1. Class `Rectangle` with `length` and `width`, methods `area()` and `perimeter()`, and a `__str__`. Create three and print each with its area and perimeter.
2. Class `Student` with `roll`, `name` and three marks; methods `total()`, `average()`, `grade()`, `display()`. Create four and print the topper's name.
3. Class `Circle` whose radius is private and exposed through a `@property`; the setter rejects a negative radius with `ValueError`. Add `area()` and `circumference()` and demonstrate a valid and an invalid assignment.
4. Create a `Bus` class inheriting from `Vehicle`. The default fare is `seating_capacity * 100`; for a bus add a further 10% as a maintenance charge. Print the fare of a car and of a bus.
5. Abstract base class `Shape` with abstract `area()`. Derive `Rectangle`, `Triangle` and `Square`, implement `area()` in each, and print all three areas from one loop.
6. Class `Account` with private `account_no`, `account_bal` and `security_code`. Initialise them through a public method (not the constructor) and print all the data through another public method. Explain in a comment why they cannot be printed directly.
7. Class `Student` displays id and name; `Marks` (derived from `Student`) displays marks for algorithms, data science and calculus; `Result` (derived from `Marks`) calculates and displays the total and average. In the driver code create one `Result` object and display everything.
8. Class `Employee` with `name`, `salary` and `calculate_bonus()` returning 10%. Derive `Manager` (20%, plus a `hire()` method) and `Developer` (plus a `write_code()` method), then `SeniorManager` from `Manager` (30%). Print every employee's name, role and bonus from one loop.
9. Class `Time` storing hours and minutes, with `__str__` printing `hh:mm` and `__add__` in which 75 minutes correctly becomes one hour fifteen. Add three times together.
10. Class `Playlist` holding song titles, with `add`, `remove`, `__len__` and a numbered `__str__`. Removing a song that is not there raises `ValueError`.
11. Multiple inheritance: `Engine` and `MusicSystem` both with a `start()` method, and `Car` inheriting from both. Print `Car().start()` and `Car.__mro__`, then swap the parent order and explain the change in a comment.
12. A vehicle-rental system: a hierarchy of `Car`, `SUV` and `Truck` with make, model, price per day and availability; methods to check availability, cost a rental of *n* days, and display details, with availability changing on rent and return. Add a `Customer` class (name, private contact number, rented vehicles, rental history) and a `RentalReservation` class linking a customer to a vehicle with start and end dates. Write one function that can display any vehicle or any reservation.

---
<div style="text-align:center;color:#666"><i>End of Lab 05 — FAST NUCES, Karachi Campus</i></div>

### `You must complete all Class and Home Tasks in the same .ipynb file, and submit using the naming format Lab 05 - Object Oriented Programming_<your_Roll_Number>.ipynb.`